In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
import operator
import os


In [2]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.7,
)

In [4]:
#Schema for the input to the LLM

class EvaluationSchema(BaseModel):
  
    feedback: str = Field(description="Feedback for the Essay")
    score: int = Field(description="Score for the Essay", ge=0, le=10)
    
    

In [5]:
structured_model = model.with_structured_output(EvaluationSchema)

In [6]:
essay ="""
The National Eligibility cum Entrance Test (NEET) is one of India's most competitive entrance examinations, serving as the gateway to medical education for millions of students. Because it determines admission to prestigious medical colleges, fairness and transparency are essential. Allegations of paper leaks have therefore raised serious concerns among students, parents, and educators.

A paper leak undermines the principle of merit. Students spend months or even years preparing honestly, sacrificing time, money, and personal comfort. When examination papers are leaked, it creates an unfair advantage for a few while diminishing the efforts of genuine aspirants. Such incidents also reduce public confidence in the examination system and place immense psychological stress on students.

The controversy has highlighted the need for stronger examination security. Experts have suggested measures such as encrypted digital distribution of question papers, stricter monitoring of the printing and transportation process, advanced surveillance at examination centres, and swift legal action against those involved in malpractice. Investigations into recent allegations have also emphasized the importance of accountability and institutional reform.

Ultimately, the integrity of competitive examinations is crucial for ensuring equal opportunities. A transparent, secure, and trustworthy examination system not only rewards hard work but also strengthens public confidence in the country's education system. Protecting the dreams of honest students must remain the highest priority.
"""

In [7]:
prompt = f"Evaluate the following essay and provide feedback and a score out of 10:\n\n{essay}"
structured_model.invoke(prompt).score

9

In [8]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[int], operator.add]
    overall_score: int
    average_score: float

In [9]:
def evaluate_language(state: UPSCState):
    prompt = f"Evaluate the language quality of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structured_model.invoke(prompt)
    
    return {
        'language_feedback': result.feedback,
        'individual_score': [result.score]
    }

In [10]:
def evaluate_analysis(state: UPSCState):
    prompt = f"Evaluate the depth analysis of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structured_model.invoke(prompt)
    
    return {
        'analysis_feedback': result.feedback,
        'individual_score': [result.score]
    }

In [11]:
def evaluate_clarity(state: UPSCState):
    prompt = f"Evaluate the clarity of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structured_model.invoke(prompt)
    
    return {
        'clarity_feedback': result.feedback,
        'individual_score': [result.score]
    }

In [ ]:
def final_evaluation(state: UPSCState):

    # Summary Feedback

    prompt = f"Provide an overall evaluation of the following essay based on the feedback received for language, analysis, and clarity. Summarize the strengths and areas for improvement:\n\nLanguage Feedback: {state['language_feedback']}\nAnalysis Feedback: {state['analysis_feedback']}\nClarity Feedback: {state['clarity_feedback']}"
    
    overall_feedback = model.invoke(prompt)

    # Average Score
    
    average_score = sum(state['individual_score']) / len(state['individual_score'])

    return {
        'overall_feedback': overall_feedback,
        'overall_score': sum(state['individual_score']),
        'average_score': average_score
    }
    

In [14]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('evaluate_overall', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_clarity')

graph.add_edge('evaluate_language', 'evaluate_overall')
graph.add_edge('evaluate_analysis', 'evaluate_overall')
graph.add_edge('evaluate_clarity', 'evaluate_overall')

graph.add_edge('evaluate_overall', END)

workflow = graph.compile()

In [16]:
initial_state = {
    'essay': essay,
    'individual_score': []
}

final_state = workflow.invoke(initial_state)

print(final_state)


{'essay': "\nThe National Eligibility cum Entrance Test (NEET) is one of India's most competitive entrance examinations, serving as the gateway to medical education for millions of students. Because it determines admission to prestigious medical colleges, fairness and transparency are essential. Allegations of paper leaks have therefore raised serious concerns among students, parents, and educators.\n\nA paper leak undermines the principle of merit. Students spend months or even years preparing honestly, sacrificing time, money, and personal comfort. When examination papers are leaked, it creates an unfair advantage for a few while diminishing the efforts of genuine aspirants. Such incidents also reduce public confidence in the examination system and place immense psychological stress on students.\n\nThe controversy has highlighted the need for stronger examination security. Experts have suggested measures such as encrypted digital distribution of question papers, stricter monitoring o